In [ ]:

# COMPLETE RAG + LANGSMITH EVALUATION

import os
import time
from dotenv import load_dotenv
from typing_extensions import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langsmith import Client, traceable
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from google.genai.errors import ClientError
from langchain_google_genai.chat_models import GoogleRateLimitError



# 1. ENVIRONMENT

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is not set")

if not LANGSMITH_API_KEY:
    raise ValueError("LANGSMITH_API_KEY is not set")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

# 2. LANGSMITH CLIENT
client = Client()

# 3. LOAD DOCUMENTS

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [
    WebBaseLoader(url).load()
    for url in urls
]

docs_list = [
    item
    for sublist in docs
    for item in sublist
]

print(f"Loaded documents: {len(docs_list)}")


# 4. SPLIT DOCUMENTS


text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50,
)

doc_splits = text_splitter.split_documents(docs_list)

print(f"Total chunks: {len(doc_splits)}")


# ============================================================
# 5. LOCAL HUGGING FACE EMBEDDINGS
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

# 6. VECTOR STORE
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)
print("Vector store created successfully.")


# 7. MAIN RAG LLM

llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=GOOGLE_API_KEY,
    max_retries=6,
)


# Helper function to invoke LLM with retry & delay for rate limits
@retry(
    wait=wait_exponential(min=4, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type((GoogleRateLimitError, ClientError, Exception)),
)
def safe_llm_invoke(chain, payload):
    time.sleep(3)  # Rate limiting buffer for 20 RPM limit
    return chain.invoke(payload)


# 8. RAG BOT

@traceable
def rag_bot(question: str) -> dict:

    docs = retriever.invoke(question)

    docs_string = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    instructions = f"""
You are a helpful RAG assistant.

Answer the user's question using ONLY the retrieved documents.

Rules:
1. Use the retrieved documents as the source of information.
2. Do not invent information.
3. If the documents do not contain enough information, say so.
4. Answer using exactly three simple bullet points.
5. Keep the answer concise.

RETRIEVED DOCUMENTS:
{docs_string}
"""

    response = safe_llm_invoke(
        llm,
        [
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ]
    )

    return {
        "answer": response.content,
        "documents": docs,
    }

# 9. CORRECTNESS EVALUATOR

class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, "Explain the reasoning for the correctness score"]
    correct: Annotated[bool, "True if the answer is correct, False otherwise"]


correctness_instructions = """
You are an expert teacher grading a quiz answer.

You will be given:
QUESTION: The question asked by the user.
GROUND TRUTH: The correct reference answer.
STUDENT ANSWER: The answer generated by the RAG system.

Evaluate ONLY factual accuracy relative to the ground truth.
Criteria:
1. Address question correctly.
2. Compare factual claims with ground truth.
3. No partial credit.

Return structured output: correct (True/False) and explanation.
"""

correctness_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=GOOGLE_API_KEY,
    max_retries=6,
).with_structured_output(
    CorrectnessGrade,
    method="json_schema",
)


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    evaluation_input = f"""
QUESTION:
{inputs["question"]}

GROUND TRUTH ANSWER:
{reference_outputs["answer"]}

STUDENT ANSWER:
{outputs["answer"]}
"""

    grade = safe_llm_invoke(
        correctness_llm,
        [
            {"role": "system", "content": correctness_instructions},
            {"role": "user", "content": evaluation_input},
        ]
    )

    return grade["correct"]


# 10. ANSWER RELEVANCE EVALUATOR

class RelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain why the answer is or is not relevant"]
    relevant: Annotated[bool, "True if the answer is relevant, False otherwise"]


relevance_instructions = """
You are an expert evaluator evaluating the relevance of an AI answer.

QUESTION: The user's question.
ANSWER: The AI-generated answer.

Evaluate ONLY relevance.
Return structured output: relevant (True/False) and explanation.
"""

relevance_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=GOOGLE_API_KEY,
    max_retries=6,
).with_structured_output(
    RelevanceGrade,
    method="json_schema",
)


def relevance(inputs: dict, outputs: dict) -> bool:
    evaluation_input = f"""
QUESTION:
{inputs["question"]}

ANSWER:
{outputs["answer"]}
"""

    grade = safe_llm_invoke(
        relevance_llm,
        [
            {"role": "system", "content": relevance_instructions},
            {"role": "user", "content": evaluation_input},
        ]
    )

    return grade["relevant"]


# 11. GROUNDEDNESS EVALUATOR

class GroundednessGrade(TypedDict):
    explanation: Annotated[str, "Explain whether the answer is supported by context"]
    grounded: Annotated[bool, "True if supported by context, False otherwise"]


groundedness_instructions = """
You are an expert evaluator checking groundedness of an AI answer against context.

Evaluate ONLY whether claims are supported by context.
Return structured output: grounded (True/False) and explanation.
"""

groundedness_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=GOOGLE_API_KEY,
    max_retries=6,
).with_structured_output(
    GroundednessGrade,
    method="json_schema",
)


def groundedness(inputs: dict, outputs: dict) -> bool:
    docs = outputs.get("documents", [])
    context = "\n\n".join(
        doc.page_content if hasattr(doc, "page_content") else str(doc)
        for doc in docs
    )

    evaluation_input = f"""
QUESTION:
{inputs["question"]}

CONTEXT:
{context}

ANSWER:
{outputs["answer"]}
"""

    grade = safe_llm_invoke(
        groundedness_llm,
        [
            {"role": "system", "content": groundedness_instructions},
            {"role": "user", "content": evaluation_input},
        ]
    )

    return grade["grounded"]


# 12. RETRIEVAL RELEVANCE EVALUATOR

class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain whether retrieved documents are relevant"]
    relevant: Annotated[bool, "True if relevant, False otherwise"]


retrieval_relevance_instructions = """
You are an expert evaluator checking if retrieved context is semantically useful.

Return structured output: relevant (True/False) and explanation.
"""

retrieval_relevance_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=GOOGLE_API_KEY,
    max_retries=6,
).with_structured_output(
    RetrievalRelevanceGrade,
    method="json_schema",
)


def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    docs = outputs.get("documents", [])
    retrieved_documents = "\n\n".join(
        doc.page_content if hasattr(doc, "page_content") else str(doc)
        for doc in docs
    )

    evaluation_input = f"""
QUESTION:
{inputs["question"]}

RETRIEVED DOCUMENTS:
{retrieved_documents}
"""
    grade = safe_llm_invoke(
        retrieval_relevance_llm,
        [
            {"role": "system", "content": retrieval_relevance_instructions},
            {"role": "user", "content": evaluation_input},
        ]
    )
    return grade["relevant"]


# 13. CREATE LANGSMITH DATASET
dataset_name = "RAG Test Evaluation"
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Using existing dataset: {dataset_name}")
except Exception:
    examples = [
        {
            "inputs": {"question": "What is an AI agent?"},
            "outputs": {
                "answer": "An AI agent is a system that can perceive a task, reason, use tools, and take actions to achieve a goal."
            },
        },
        {
            "inputs": {"question": "What is the ReAct pattern in AI agents?"},
            "outputs": {
                "answer": "ReAct combines reasoning and action. The agent reasons, acts, observes, and repeats until complete."
            },
        },
        {
            "inputs": {"question": "Why do AI agents need tools?"},
            "outputs": {
                "answer": "Tools enable agents to perform web search, query databases, run code, and invoke APIs."
            },
        },
        {
            "inputs": {"question": "What is tool calling in an AI agent?"},
            "outputs": {
                "answer": "Tool calling allows an LLM to request function execution with structured arguments."
            },
        },
        {
            "inputs": {"question": "What is the difference between an AI agent and a chatbot?"},
            "outputs": {
                "answer": "Chatbots produce conversational replies; agents reason, use tools, and take autonomous actions."
            },
        },
        {
            "inputs": {"question": "What is RAG?"},
            "outputs": {
                "answer": "RAG retrieves relevant external information to augment LLM context before generation."
            },
        },
        {
            "inputs": {"question": "What are the main steps in a RAG pipeline?"},
            "outputs": {
                "answer": "Loading docs, chunking, embedding, vector storage, context retrieval, and generation."
            },
        },
        {
            "inputs": {"question": "What are embeddings in RAG?"},
            "outputs": {
                "answer": "Embeddings are vector representations capturing semantic text meaning for similarity search."
            },
        },
    ]

    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset: {dataset_name}")


# 14. TARGET FUNCTION

def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])


# 15. RUN LANGSMITH EVALUATION (With Concurrency Control)=
print("\nStarting LangSmith evaluation...\n")
experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness,
        groundedness,
        relevance,
        retrieval_relevance,
    ],
    max_concurrency=1,  # Key fix: Runs sequentially to avoid hitting 20 RPM
    experiment_prefix="rag-doc-relevance",
    metadata={
        "version": "Gemini 2.5 Flash Lite + HuggingFace Embeddings",
        "evaluation": "correctness, groundedness, relevance, retrieval_relevance",
    },
)


# 16. RESULTS

results_df = experiment_results.to_pandas()

print("\nEvaluation completed successfully.")
print(results_df)

Loaded documents: 3
Total chunks: 91


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4491.30it/s]


Vector store created successfully.
Using existing dataset: RAG Test Evaluation

Starting LangSmith evaluation...

View the evaluation results for experiment: 'rag-doc-relevance-8c23f224' at:
https://smith.langchain.com/o/1dedea63-72fd-48b3-b04c-058788302da7/datasets/54865e1a-b429-4723-becb-d26a5463a79b/compare?selectedSessions=37582fcd-5afb-49a3-acf6-ff3a8ce9b507




0it [00:00, ?it/s]Error running target function: RetryError[<Future at 0x76dbf52a6990 state=finished raised GoogleRateLimitError>]
Traceback (most recent call last):
  File "/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py", line 4177, in _generate
    response: GenerateContentResponse = self.client.models.generate_content(
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/google/genai/models.py", line 6270, in generate_content
    response = self._generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/google/genai/models.py", line 4707, in _generate_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/google/genai/_api_client.py", line 1750, in request
    response = self